# Notebook 13 — Camera Geometry & Calibration

For every major concept: **Intuition → Straight definition → Example**. The important mathematics is explained carefully.


## 1. Why Camera Geometry?

**Intuition:** A camera turns a 3D world into a 2D image. We need mathematics to understand that conversion.

**Straight definition:** Camera geometry describes the relationship between 3D points and their 2D image positions.

**Example:** A 3D point `(X,Y,Z)` becomes an image point `(u,v)`.


## 2. Pinhole Camera Model

**Intuition:** Imagine light travelling from a 3D object through a tiny hole onto an image plane.

**Straight definition:** The pinhole model projects 3D coordinates using perspective projection.

`x = fX/Z`

`y = fY/Z`

**Example:** If `X=2, Z=4, f=2`, then `x=1`.


In [ ]:
X, Y, Z = 2, 3, 4
f = 2
x = f * X / Z
y = f * Y / Z
print("Projected point:", (x, y))


## 3. Perspective Projection

**Intuition:** Far objects look smaller because their projection is compressed by depth.

**Straight definition:** Perspective projection divides horizontal and vertical coordinates by depth.

`x = fX/Z`, `y = fY/Z`.

**Example:** Doubling `Z` halves the projected size.


## 4. Camera Intrinsic Matrix

**Intuition:** The camera needs to know its focal length and image center.

**Straight definition:** The intrinsic matrix contains internal camera parameters.

`K = [[fx,0,cx],[0,fy,cy],[0,0,1]]`

`fx,fy` are focal lengths in pixels; `cx,cy` are the principal point.

**Example:** A 640×480 image often has a principal point near `(320,240)`.


In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

K = np.array([[800,0,320],[0,800,240],[0,0,1]], dtype=np.float32)
print(K)


## 5. Intrinsic vs Extrinsic Parameters

**Intuition:** Intrinsics describe the camera itself; extrinsics describe where it is and how it is rotated.

**Straight definition:** Intrinsics are `fx,fy,cx,cy`; extrinsics are rotation `R` and translation `t`.

**Example:** Moving the camera changes `R,t`, not the lens intrinsics.


## 6. Camera Projection Equation

`x_image ~ K [R | t] X_world`

**Intuition:** First transform the 3D point into the camera coordinate system, then project it into the image.

**Straight definition:** `K` handles internal camera geometry while `[R|t]` handles the camera pose.

`~` means equality up to scale.


## 7. Coordinate Systems

**Intuition:** The same physical point can have different coordinates depending on the reference frame.

**Straight definition:** Common frames are world `(X,Y,Z)`, camera `(Xc,Yc,Zc)`, and image `(u,v)`.

**Example:** A point can be `(2,1,5)` in world coordinates but have a different coordinate after the camera moves.


## 8. Camera Distortion

**Intuition:** Real lenses are imperfect, so straight lines near the image edges may appear curved.

**Straight definition:** Distortion is the difference between ideal pinhole projection and the real lens projection.

**Example:** Wide-angle lenses commonly show barrel distortion.


## 9. Radial Distortion

**Intuition:** The farther a point is from the image center, the more the lens can bend it.

**Straight definition:** Radial distortion changes coordinates according to their distance from the optical center.

`x_d = x(1 + k1*r² + k2*r⁴ + ...)`

`y_d = y(1 + k1*r² + k2*r⁴ + ...)`

where `r²=x²+y²`.


## 10. Tangential Distortion

**Intuition:** If the lens is not perfectly aligned with the sensor, points can shift sideways.

**Straight definition:** Tangential distortion models lens/sensor misalignment.

`x_d = x + 2p1xy + p2(r² + 2x²)`

`y_d = y + p1(r² + 2y²) + 2p2xy`


## 11. What is Camera Calibration?

**Intuition:** Give OpenCV several views of a known pattern and let it estimate the camera model.

**Straight definition:** Calibration estimates the camera matrix, distortion coefficients, and camera poses.

**Example:** A chessboard with known square size is commonly used.


## 12. Chessboard Calibration Pattern

**Intuition:** The chessboard gives us many points whose real-world positions are known.

**Straight definition:** Calibration matches known 3D object points to detected 2D image points. For a flat board, `Z=0`.

**Example:** A `(9,6)` pattern means 54 internal corners.


In [ ]:
pattern_size = (9, 6)
object_points = np.zeros((pattern_size[0]*pattern_size[1], 3), np.float32)
object_points[:, :2] = np.mgrid[0:pattern_size[0], 0:pattern_size[1]].T.reshape(-1,2)
print("Number of points:", len(object_points))
print(object_points[:5])


## 13. Finding Chessboard Corners

**Intuition:** OpenCV can automatically find the chessboard corners instead of making us mark them manually.

**Straight definition:** `findChessboardCorners()` detects internal chessboard corners.

**Syntax:** `ret, corners = cv2.findChessboardCorners(gray, (9,6))`

**Example:** Use it on each calibration image.


In [ ]:
image = cv2.imread("calibration_01.jpg")
if image is not None:
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    ret, corners = cv2.findChessboardCorners(gray, (9,6), None)
    print("Corners found:", ret)
else:
    print("Add calibration_01.jpg to run this example.")


## 14. Refining Corners

**Intuition:** The first detected corner location is close, but we can make it more precise.

**Straight definition:** `cornerSubPix()` refines corner coordinates to sub-pixel accuracy.

**Example:** Better corner coordinates improve calibration.


In [ ]:
criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 0.001)
print("Use cv2.cornerSubPix(gray, corners, (11,11), (-1,-1), criteria)")


## 15. Camera Calibration

**Intuition:** Collect known chessboard points and detected image points from several images, then solve for the camera parameters.

**Straight definition:** `cv2.calibrateCamera()` estimates `K`, distortion, rotation vectors, and translation vectors.

**Syntax:** `rms, K, dist, rvecs, tvecs = cv2.calibrateCamera(...)`


## 16. Complete Calibration Template

Put several files such as `calibration_01.jpg`, `calibration_02.jpg`, etc. in the notebook folder.


In [ ]:
import glob
pattern_size = (9, 6)
square_size = 1.0
objp = np.zeros((pattern_size[0]*pattern_size[1],3), np.float32)
objp[:, :2] = np.mgrid[0:pattern_size[0],0:pattern_size[1]].T.reshape(-1,2) * square_size
object_points, image_points = [], []
for path in glob.glob("calibration_*.jpg"):
    img = cv2.imread(path)
    if img is None: continue
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    found, corners = cv2.findChessboardCorners(gray, pattern_size, None)
    if found:
        object_points.append(objp)
        image_points.append(corners)
if image_points:
    rms, K, dist, rvecs, tvecs = cv2.calibrateCamera(object_points, image_points, gray.shape[::-1], None, None)
    print("RMS error:", rms)
    print("Camera matrix:\n", K)
    print("Distortion:", dist.ravel())
else:
    print("No calibration images found.")


## 17. Reprojection Error

**Intuition:** Project the known 3D points back into the image. Good calibration makes the predicted corners land close to the real corners.

**Straight definition:** Reprojection error is the image-space distance between observed and predicted points.

`e = sqrt((u_obs-u_pred)^2 + (v_obs-v_pred)^2)`

**Example:** `0.3 px` is much better than `8 px`.


## 18. Undistortion

**Intuition:** Once distortion is known, reverse it to obtain a geometrically corrected image.

**Straight definition:** `cv2.undistort()` removes estimated lens distortion.

**Example:** Curved lines near the edges can become straight.


In [ ]:
# After calibration:
# image = cv2.imread("calibration_01.jpg")
# undistorted = cv2.undistort(image, K, dist)
print("cv2.undistort(image, K, dist)")


## 19. Why Multiple Calibration Images?

**Intuition:** One view cannot provide enough geometric variety.

**Straight definition:** Multiple views constrain the camera parameters more reliably.

Use different positions, rotations, distances, and image regions.

**Example:** 20 nearly identical images are less useful than 20 varied views.


## 20. Pose Estimation

**Intuition:** If we know what an object looks like in 3D and where its points appear in the image, we can estimate how it is positioned relative to the camera.

**Straight definition:** Pose estimation finds rotation and translation from known 3D–2D correspondences.

**Example:** Estimate the pose of a chessboard or AR marker.


## 21. solvePnP

**Intuition:** Give OpenCV known 3D points, their 2D image locations, and the calibrated camera. It finds the pose that explains them.

**Straight definition:** `cv2.solvePnP()` estimates rotation and translation.

**Syntax:** `success, rvec, tvec = cv2.solvePnP(objectPoints, imagePoints, K, distCoeffs)`


In [ ]:
print("cv2.solvePnP(objectPoints, imagePoints, K, distCoeffs)")


## 22. Rotation Vector

**Intuition:** A rotation vector is a compact representation of 3D rotation.

**Straight definition:** OpenCV uses a Rodrigues rotation vector `rvec`; `cv2.Rodrigues()` converts it to a 3×3 rotation matrix.

**Example:**


In [ ]:
rvec = np.array([[0.2],[0.1],[0.0]], dtype=np.float32)
R, _ = cv2.Rodrigues(rvec)
print(R)


## 23. Projecting 3D Points

**Intuition:** If camera parameters and pose are known, we can predict where a 3D point appears in the image.

**Straight definition:** `cv2.projectPoints()` maps 3D object points to 2D image points.

**Example:** Project the corners of a virtual cube.


In [ ]:
points_3d = np.array([[0,0,0],[1,0,0],[1,1,0],[0,1,0]], np.float32)
rvec = np.zeros((3,1), np.float32)
tvec = np.array([[0],[0],[5]], np.float32)
print("cv2.projectPoints(points_3d, rvec, tvec, K, dist)")


## 24. Camera Calibration Pipeline

**Intuition:** `Chessboard → corners → known 3D points → 3D/2D correspondences → calibration → K + distortion → undistortion`.

**Straight definition:** This is the standard workflow for estimating a camera model from a calibration target.


## 25. Important Functions Cheat Sheet

| Function | Purpose |
|---|---|
| `findChessboardCorners()` | Detect chessboard corners |
| `cornerSubPix()` | Refine corners |
| `calibrateCamera()` | Estimate camera parameters |
| `undistort()` | Remove distortion |
| `getOptimalNewCameraMatrix()` | Improve usable image area |
| `solvePnP()` | Estimate pose |
| `Rodrigues()` | Rotation vector ↔ matrix |
| `projectPoints()` | 3D → 2D projection |
| `drawChessboardCorners()` | Visualize corners |


## 26. Practice Tasks

1. Detect chessboard corners.
2. Collect several chessboard images and calibrate a webcam.
3. Print `K` and distortion coefficients.
4. Undistort one image.
5. Calculate reprojection error.
6. Use `solvePnP()` to estimate pose.
7. Use `projectPoints()` to draw a virtual 3D cube.


# Summary

- **Pinhole model:** 3D → 2D perspective projection.
- **Intrinsics:** internal camera geometry.
- **Extrinsics:** camera/object pose.
- **Distortion:** real-lens imperfections.
- **Calibration:** estimates `K`, distortion, and poses.
- **Undistortion:** corrects lens distortion.
- **solvePnP:** estimates pose.
- **projectPoints:** projects 3D geometry into an image.

This is the foundation for AR, 3D reconstruction, stereo vision, and robotics.
